In [1]:
pip install polars


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import random
import string
import time

# Function to generate random strings
def random_string(length=10):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

# Function to generate random dates
def random_date(start='2010-01-01', end='2025-12-31'):
    return pd.to_datetime(pd.to_datetime(start) + np.random.rand() * (pd.to_datetime(end) - pd.to_datetime(start)))

# Number of rows to generate (adjust for file size)
num_rows = 10**5  # ~1 million rows; adjust as needed for your file size

# Generate a large dataset
data = {
    'id': range(1, num_rows + 1),
    'name': [random_string(10) for _ in range(num_rows)],
    'date_of_birth': [random_date() for _ in range(num_rows)],
    'city': [random_string(5) for _ in range(num_rows)],
    'income': np.random.randint(30000, 150000, size=num_rows),
    'status': [random.choice(['Single', 'Married', 'Divorced', 'Widowed']) for _ in range(num_rows)],
}

# Create a DataFrame
df = pd.DataFrame(data)

# Specify the path to save the CSV file
file_path = r'C:\Users\Notebook\Jupyter Python Notebooks\large_test_file.csv'

# Measure the time taken to save the file
start_time = time.time()
df.to_csv(file_path, index=False)
end_time = time.time()

print(f"CSV file generated at: {file_path}")
print(f"Time taken to generate the file: {end_time - start_time:.2f} seconds")

# You can change num_rows to generate a larger file if needed.

CSV file generated at: C:\Users\Notebook\Jupyter Python Notebooks\large_test_file.csv
Time taken to generate the file: 0.44 seconds


In [6]:
import polars as pl
import numpy as np
import string
import os
import time

file_path = r"C:\Users\Notebook\Jupyter Python Notebooks\large_test_file.csv"

rows_per_chunk = 1_000_000
num_chunks = 25

if os.path.exists(file_path):
    os.remove(file_path)

def random_strings(n, length):
    letters = np.array(list(string.ascii_letters))
    return (
        np.random.choice(letters, (n, length))
        .view(f'U{length}')
        .reshape(n)
    )

start_total = time.time()

for chunk in range(num_chunks):

    start_id = chunk * rows_per_chunk
    end_id = start_id + rows_per_chunk

    df = pl.DataFrame({
        "id": np.arange(start_id, end_id),
        "name": random_strings(rows_per_chunk, 10),
        "city": random_strings(rows_per_chunk, 6),
        "income": np.random.randint(30000, 150000, rows_per_chunk),
        "status": np.random.choice(
            ["Single", "Married", "Divorced", "Widowed"],
            rows_per_chunk
        )
    })

    if chunk == 0:
        # First chunk creates file + header
        df.write_csv(file_path)
    else:
        # Append in binary mode (IMPORTANT)
        with open(file_path, "ab") as f:
            df.write_csv(f, include_header=False)

    print(f"Chunk {chunk+1}/{num_chunks} written")

end_total = time.time()

print("\nFinished!")
print(f"Total generation time: {end_total - start_total:.2f} seconds")

Chunk 1/25 written
Chunk 2/25 written
Chunk 3/25 written
Chunk 4/25 written
Chunk 5/25 written
Chunk 6/25 written
Chunk 7/25 written
Chunk 8/25 written
Chunk 9/25 written
Chunk 10/25 written
Chunk 11/25 written
Chunk 12/25 written
Chunk 13/25 written
Chunk 14/25 written
Chunk 15/25 written
Chunk 16/25 written
Chunk 17/25 written
Chunk 18/25 written
Chunk 19/25 written
Chunk 20/25 written
Chunk 21/25 written
Chunk 22/25 written
Chunk 23/25 written
Chunk 24/25 written
Chunk 25/25 written

Finished!
Total generation time: 32.40 seconds


In [7]:
import pandas as pd
import time

start = time.time()
df_pd = pd.read_csv(file_path)
print("Pandas load time:", time.time() - start)

Pandas load time: 184.8075041770935


In [8]:
import polars as pl
import time

start = time.time()
df_pl = pl.read_csv(file_path)
print("Polars load time:", time.time() - start)

Polars load time: 9.150943040847778


In [9]:
import pandas as pd
import polars as pl
import time

file_path = r"C:\Users\Notebook\Jupyter Python Notebooks\large_test_file.csv"

print("=== Pandas Eager ===")
start = time.time()
df_pd = pd.read_csv(file_path)
# Example operation: filter income > 50k and select columns
df_pd = df_pd[df_pd["income"] > 50000][["id", "income"]]
end = time.time()
print(f"Pandas time: {end - start:.2f} s")
print(f"Rows: {len(df_pd)}\n")

print("=== Polars Eager ===")
start = time.time()
df_pl = pl.read_csv(file_path)
# Same operation
df_pl = df_pl.filter(pl.col("income") > 50000).select(["id", "income"])
end = time.time()
print(f"Polars eager time: {end - start:.2f} s")
print(f"Rows: {df_pl.height}\n")

print("=== Polars Lazy ===")
start = time.time()
df_lazy = pl.scan_csv(file_path)  # Lazy read
df_lazy = df_lazy.filter(pl.col("income") > 50000).select(["id", "income"])
# Trigger execution
df_lazy_result = df_lazy.collect()
end = time.time()
print(f"Polars lazy time: {end - start:.2f} s")
print(f"Rows: {df_lazy_result.height}")

=== Pandas Eager ===
Pandas time: 143.22 s
Rows: 20834886

=== Polars Eager ===
Polars eager time: 9.09 s
Rows: 20834886

=== Polars Lazy ===
Polars lazy time: 1.84 s
Rows: 20834886
